### Master of Applied Artificial Intelligence

**Course: TC5035 - Proyecto Integrador**

<img src="https://github.com/Medicenchapin/Proyecto-Integrador/blob/main/assets/logo.png?raw=1" alt="Image Alt Text" width="500"/>


**Baseline model**

Tutor: Dr. Horario Martinez Alfaro


Team members:
* Ignacio Jose Aguilar Garcia - A00819762
* Alejandro Calderon Aguilar - A01795353
* Ricardo Mar Cupido - A01795394

## Import

Load standard libraries

In [3]:
import sys
sys.path.append('../')

In [4]:
import importlib, scripts.helpers as hp
importlib.reload(hp)
from scripts.helpers import Helpers
import pandas as pd
import requests
import os


We load the dataset extracted from the final model used throughout the course.

In [5]:
df = pd.read_parquet("../data/campaign_candidates_final.parquet")

## EDA

We performed exploratory data analysis using head, tail, shape, info and describe. These steps help identify missing values, outliers, cardinality, and biases, and they guide the corrective actions we should take.

The table shown below is built from the raw dataset and contains the features that most influenced the model’s per-customer feature importance (SHAP-derived drivers). For each customer we capture the top drivers and their metrics to support business-facing insights.

Key fields included in the table:

- previous_classification: Customer commercial tag (for example: NEW_CLIENT, NOT_INTERESTED, NOT_EFFECTIVE). Use this field to adjust the communication tone — onboarding, retention, or reactivation.

- previous_calls: Number of previous calls/interactions with the customer. A high count may indicate either sustained interest or unresolved friction; adapt the approach depending on the driver direction.

- driver: Extraction of the per-customer driver list (SHAP-style). Each entry in this JSON-like column typically contains: feature (name), impact (signed contribution), raw_feature (original column name), raw_value (original untransformed value), and value (the transformed numeric value used by the model).

Notes:
- We present the top N drivers per customer (usually top 5) to produce concise, evidence-based agent scripts.
- When preparing agent-facing text, surface only raw evidence and actionable insights (e.g., “Used 5.9 GB of music streaming last period”) and avoid exposing technical internals (model weights, SHAP algorithm names) in the final agent script.


In [16]:
df['previous_classification'].unique()

array(['NEW CLIENT', 'NOT EFFECTIVE', 'NOT INTERESTED'], dtype=object)

In [17]:
df['previous_calls'].unique()

array([ 0,  2,  1,  3,  4,  5,  7,  6,  8,  9, 11])

In [15]:
df.head()

,state_name,previous_classification,previous_calls,client_age,network_age_years,banking,arpu_90_days,minutes_in,validity_average,average_performance,...,plan_postpaid,sn_banking,digital_index_mean,connected_days,charged_days,apps_days,music_gb,proba,sample_idx,drivers
0,GUATEMALA,NEW CLIENT,0,28.0,3.90,1,214.18,27.93,20.23,0.46,...,0.67,1.00,0.00,91,91,0,5.946,0.699237,0,"[{'feature': 'contacts', 'impact': -0.58204424..."
1,GUATEMALA,NEW CLIENT,0,19.0,0.66,0,92.58,48.77,3.14,0.61,...,0.75,1.00,0.00,76,55,4,0.031,0.781733,10,"[{'feature': 'contacts', 'impact': -0.43256592..."
2,SAN MARCOS,NEW CLIENT,0,20.0,0.34,1,175.49,83.58,4.00,0.25,...,0.12,0.56,0.00,91,59,30,0.000,0.641149,12,"[{'feature': 'client_age', 'impact': 0.4716295..."
3,SAN MARCOS,NEW CLIENT,0,NaN,1.90,1,97.54,51.89,8.36,0.77,...,0.54,0.38,0.29,90,77,11,0.000,0.690091,26,"[{'feature': 'plan_postpaid', 'impact': 0.5216..."
4,SAN MARCOS,NOT EFFECTIVE,2,31.0,5.87,1,102.25,78.93,15.00,0.69,...,0.24,0.30,0.00,89,64,22,0.000,0.733312,37,"[{'feature': 'contacts', 'impact': 0.383831799..."


In [8]:
df.tail()

,state_name,previous_classification,previous_calls,client_age,network_age_years,banking,arpu_90_days,minutes_in,validity_average,average_performance,...,plan_postpaid,sn_banking,digital_index_mean,connected_days,charged_days,apps_days,music_gb,proba,sample_idx,drivers
18298,GUATEMALA,NEW CLIENT,0,22.0,3.16,1,103.42,38.98,2.00,0.70,...,0.50,0.75,0.89,91,91,0,0.334,0.694028,182992,"[{'feature': 'client_age', 'impact': 0.4018475..."
18299,GUATEMALA,NEW CLIENT,0,39.0,0.22,1,101.54,21.88,2.60,0.51,...,0.79,0.86,0.32,75,56,3,0.000,0.756829,182993,"[{'feature': 'plan_postpaid', 'impact': 0.4847..."
18300,JUTIAPA,NEW CLIENT,0,40.0,0.24,0,92.20,12.46,14.00,0.73,...,0.43,0.43,0.00,55,50,2,0.000,0.628511,182998,"[{'feature': 'plan_postpaid', 'impact': 0.5019..."
18301,PETEN,NOT EFFECTIVE,2,NaN,10.57,1,100.21,42.05,7.38,0.55,...,0.27,0.45,0.56,91,67,13,0.481,0.675360,183004,"[{'feature': 'plan_postpaid', 'impact': 0.2827..."
18302,GUATEMALA,NEW CLIENT,0,NaN,0.23,0,109.61,79.16,5.38,0.73,...,0.56,0.69,0.00,79,69,6,0.000,0.805487,183012,"[{'feature': 'plan_postpaid', 'impact': 0.5441..."


In [9]:
df.shape

(18303, 23)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18303 entries, 0 to 18302
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   state_name               18303 non-null  object 
 1   previous_classification  18303 non-null  object 
 2   previous_calls           18303 non-null  int64  
 3   client_age               15198 non-null  float64
 4   network_age_years        18302 non-null  float64
 5   banking                  18303 non-null  int64  
 6   arpu_90_days             18303 non-null  float64
 7   minutes_in               18303 non-null  float64
 8   validity_average         18303 non-null  float64
 9   average_performance      18303 non-null  float64
 10  start_using_months       18303 non-null  float64
 11  contacts                 18303 non-null  int64  
 12  high_frequency_contacts  18303 non-null  float64
 13  plan_postpaid            18303 non-null  float64
 14  sn_banking            

In [11]:
df.describe()

,previous_calls,client_age,network_age_years,banking,arpu_90_days,minutes_in,validity_average,average_performance,start_using_months,contacts,high_frequency_contacts,plan_postpaid,sn_banking,digital_index_mean,connected_days,charged_days,apps_days,music_gb,proba,sample_idx
count,18303.000000,15198.000000,18302.000000,18303.000000,18303.000000,18303.000000,18303.000000,18303.000000,18303.000000,18303.000000,18303.000000,18303.000000,18303.000000,18303.000000,18303.000000,18303.000000,18303.000000,18303.000000,18303.000000,18303.000000
mean,0.616019,35.076655,2.851063,0.796427,119.449417,88.766071,7.342006,0.656964,5.326231,59.117795,0.481439,0.355543,0.514961,0.301608,83.111785,73.398623,4.884828,0.626546,0.683225,90895.709173
std,0.954471,11.291008,3.333412,0.402666,33.516896,121.683255,6.307007,0.170048,7.325909,43.232384,0.116029,0.194927,0.208324,0.378933,10.789578,15.746811,7.097796,1.798736,0.051580,52746.124613
min,0.000000,18.000000,0.070000,0.000000,62.280000,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.617508,0.000000
25%,0.000000,26.000000,0.440000,1.000000,99.260000,25.610000,2.840000,0.540000,1.000000,30.000000,0.410000,0.210000,0.360000,0.000000,79.000000,64.000000,0.000000,0.000000,0.641749,45598.000000
50%,0.000000,33.000000,1.680000,1.000000,107.780000,54.100000,5.620000,0.670000,2.000000,47.000000,0.480000,0.330000,0.490000,0.000000,88.000000,77.000000,2.000000,0.000000,0.671460,90641.000000
75%,1.000000,42.000000,3.937500,1.000000,127.100000,108.515000,9.485000,0.780000,6.000000,76.000000,0.550000,0.470000,0.650000,0.640000,91.000000,87.000000,7.000000,0.533000,0.713606,136482.000000
max,11.000000,89.000000,22.630000,1.000000,515.720000,5610.910000,30.000000,0.990000,53.000000,450.000000,1.000000,1.000000,1.000000,1.000000,91.000000,91.000000,65.000000,44.895000,0.912042,183012.000000


In [14]:
df.describe(include='object')

,state_name,previous_classification,drivers
count,18303,18303,18303
unique,22,3,18303
top,GUATEMALA,NEW CLIENT,"[{'feature': 'contacts', 'impact': -0.58204424..."
freq,5695,11233,1


In [13]:
df.columns

Index(['state_name', 'previous_classification', 'previous_calls', 'client_age',
       'network_age_years', 'banking', 'arpu_90_days', 'minutes_in',
       'validity_average', 'average_performance', 'start_using_months',
       'contacts', 'high_frequency_contacts', 'plan_postpaid', 'sn_banking',
       'digital_index_mean', 'connected_days', 'charged_days', 'apps_days',
       'music_gb', 'proba', 'sample_idx', 'drivers'],
      dtype='object')

In [7]:
df['drivers'][0]

array([{'feature': 'contacts', 'impact': -0.582044243812561, 'ohe_category': None, 'raw_feature': 'contacts', 'raw_value': '8', 'value': -1.8448108052715728},
       {'feature': 'music_gb', 'impact': 0.31890058517456055, 'ohe_category': None, 'raw_feature': 'music_gb', 'raw_value': '5.946', 'value': 5.946},
       {'feature': 'plan_postpaid', 'impact': 0.3081912696361542, 'ohe_category': None, 'raw_feature': 'plan_postpaid', 'raw_value': '0.6699999999999999', 'value': 1.869873907209015},
       {'feature': 'client_age', 'impact': 0.18900930881500244, 'ohe_category': None, 'raw_feature': 'client_age', 'raw_value': '28.0', 'value': -1.1497417570435924},
       {'feature': 'average_performance', 'impact': 0.17730332911014557, 'ohe_category': None, 'raw_feature': 'average_performance', 'raw_value': '0.4599999999999999', 'value': -1.2696643851806635}],
      dtype=object)

## instance of helpers

In [39]:
helpers = Helpers(df=df)

## Build global prompt

In [40]:
global_prompt = helpers.build_global_system_prompt_es()
print(global_prompt)

Eres un asistente analítico para una empresa de telecomunicaciones. Tu función es ayudar a interpretar los principales drivers (valores SHAP) del modelo a nivel global y por cliente, en términos de negocio.

            Resumen Global de Drivers SHAP
            Estas son las variables globalmente más influyentes (TOP 10) y su significado de negocio:
            - plan_postpaid: Indica si el cliente tiene un plan pospago (1 = sí, 0 = no). Los clientes pospago valoran la experiencia premium, estabilidad y confiabilidad.
- music_gb: Cantidad de datos móviles (en GB) usados para música. Un valor cero representa oportunidad para ofertas de 'música sin consumo de datos'.
- client_age: Edad del cliente en años. Evita sesgos demográficos; úsala solo para ajustar el tono de comunicación si es necesario.
- contacts: Total de contactos previos con la empresa. Si es alto y el SHAP es negativo, sugiere reducir fricción y simplificar procesos.
- network_age_years: Años desde que el cliente se unió 

## Build prompt query by customer

In [8]:
idx = 26
row = df.loc[idx]

prompt_for_customer = helpers.build_customer_prompt_summary(
    row=row,
    driver_list=row["drivers"]
)

print(prompt_for_customer)

Eres un analista de campañas de telecomunicaciones prepago.

        Analiza los factores más influyentes en la probabilidad de compra para un cliente individual, basándote en valores SHAP.
        El modelo predijo una probabilidad de aceptación del **66.4%**.

        A continuación se listan los principales *drivers* (variables) que explican esta predicción,
        ordenados por relevancia:

        - **plan_postpaid** (positivo (favorece contacto)): valor = 2.03. Indica si el cliente tiene un plan pospago (1 = sí, 0 = no). Los clientes pospago valoran la experiencia premium, estabilidad y confiabilidad.
- **contacts** (negativo (revisar antes de contactar)): valor = -1.40. Total de contactos previos con la empresa. Si es alto y el SHAP es negativo, sugiere reducir fricción y simplificar procesos.
- **arpu_90_days** (positivo (favorece contacto)): valor = -1.21. Ingreso promedio por usuario en los últimos 90 días. Un ARPU alto indica cliente activo o de alto valor; un ARPU bajo sug

In [9]:

import requests
import os
import ollama

OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434/api/generate")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "cas/nous-hermes-2-mistral-7b-dpo")

def call_llm(global_context: str, customer_prompt: str) -> str:
    full_prompt = [
        {'role': 'system', 'content': global_context},
        {'role': 'user', 'content': customer_prompt}
    ]
    out: str = ''
    for chunk in ollama.chat(OLLAMA_MODEL, messages=full_prompt, stream=True):
        print(chunk['message']['content'], end='', flush=True)
    # return out.strip()

In [10]:
call_llm(global_context=global_prompt, customer_prompt=prompt_for_customer)

1. El cliente puede estar motivado por ser nuevo en nuestra empresa (previous_classification_NEW CLIENT) y tener un ingreso promedio alto en los últimos 90 días (arpu_90_days). Estos factores sugieren que es un cliente activo o de alto valor. Por otro lado, el alto número de contactos anteriores con la empresa (contacts) podría indicar fricción o complejidad en nuestros procesos, lo que debe ser evaluado y simplificado si es necesario.

2. Considerando los valores SHAP, convendría analizar más información antes de contactar al cliente, ya que algunos factores pueden requerir atención o ajustes en nuestras operaciones. La probabilidad de aceptación es del 66.4%, lo que implica que podríamos ganar su confianza si abordamos las posibles áreas de mejora y ofrecemos una experiencia óptima.

3. No menciono palabras técnicas como "modelo", "SHAP", "algoritmo" o "predicción".

4. En lenguaje de negocio claro y objetivo, con tono ejecutivo: Este cliente es probablemente un individuo activo e im

In [ ]:
idx_2 = 12
row = df.loc[idx]

# prompt_for_customer_2 = helpers.build_customer_prompt_with_shap(
#     row=row,
#     driver_list=row["drivers"]
# )


prompt_customer2 = helpers.build_customer_prompt_with_shap(
    row=row,
    driver_list=row["drivers"],
    # max_features=10,
    # include_json_mirror=True  # to request a mirrored JSON in addition to the bullet points
)

In [42]:
print(prompt_customer2)

Eres un asistente analítico para campañas de telecomunicaciones prepago.

    Cliente:
    - Probabilidad estimada de aceptación: 66.4%

    A continuación tienes los **10 principales drivers SHAP** del cliente (ya ordenados por relevancia).
    Cada elemento incluye: nombre para mostrar, valor crudo, valor transformado, valor SHAP y una pista de negocio.
    **No inventes ni alteres valores numéricos**: utiliza exactamente los provistos.

    DRIVERS_JSON:
    [{"feature_display": "plan_postpaid", "feature": "plan_postpaid", "raw_feature": "plan_postpaid", "ohe_category": null, "raw_value": "0.8", "transformed_value": 2.030105, "shap_value": 0.460809, "direction": "positivo", "business_hint": "Indica si el cliente tiene un plan pospago (1 = sí, 0 = no). Los clientes pospago valoran la experiencia premium, estabilidad y confiabilidad."}, {"feature_display": "contacts", "feature": "contacts", "raw_feature": "contacts", "ohe_category": null, "raw_value": "13", "transformed_value": -1.401

In [43]:
call_llm(global_context=global_prompt, customer_prompt=prompt_customer2)

### Resumen SHAP del Cliente

* contacts: negativo. SHAP=0.383442. Crudo=13. Transformado=-1.401177. El número de contactos anteriores es alto, lo que sugiere reducir fricción y simplificar procesos.
* arpu_90_days: positivo. SHAP=0.22351. Crudo=97.78. Transformado=-1.205831. Un ARPU alto indica un cliente activo o de alto valor, lo que puede generar mayores ingresos a través de promociones personalizadas.
* previous_classification_NEW CLIENT: positivo. SHAP=0.122369. Crudo=null. Transformado=1.0. El cliente es nuevo, lo que puede generar oportunidades de cruce y venta de productos complementarios.
* state_name: positivo. SHAP=0.11969. Cruda=GUATEMALA. Transformado=0.044849. La ubicación en Guatemala puede ser utilizada para promocionar cobertura y servicios locales.
* contacts: negativo. SHAP=-0.383442. Crudo=13. Transformado=-1.401177. El número de contactos anteriores es alto, lo que sugiere reducir fricción y simplificar procesos.
* arpu_90_days: positivo. SHAP=0.22351. Crudo=97.78

---

## Conclusion:

* In the first attempt, the prompt generated a full sales script for the agent (indicating which plan to sell, the probability of sale and a price recommendation). This forced the agent to read several paragraphs and reduced operational speed.

* Feedback from the manager (Martín): request only bullet points with the most relevant factors per customer so the agent reads less and processes the list with much greater fluency.

* Change applied: we adapted the prompt to return five bullets per customer using the format "Feature | Impact | Raw Value | Short insight/action", prioritizing evidence (raw_value) and avoiding technical details in the agent script.

* Initial result: the bullet version was well received by the manager and considered suitable to move forward toward a pilot implementation, because it reduces the agent’s cognitive load and enables higher-volume execution.

* Next recommended steps: run a controlled A/B pilot to measure impact on contact and conversion rates, and produce a structured JSON output for direct integration with CRM/operations.

Result of the output you sent (summary + why it’s good)

What the output showed: the driver payload you provided highlights a small set of clear signals (contacts negative, arpu_90_days positive, previous_classification = NEW_CLIENT positive, state_name positive). The payload included duplicates which we deduplicated; after that we produced the compact evidence bullets and a short recommendation JSON.

Why this is a good outcome: each bullet is evidence-first (shows the raw metric), includes a clear direction (impact sign), and ends with a single, business-focused insight or action. That structure lets agents read one line per driver and immediately understand (a) the data point, (b) whether it helps or hurts conversion, and (c) the recommended handling — all without technical jargon.

Why each bullet’s feedback is valuable and easy to assimilate

contacts (negative): “High prior contacts → short, low-friction offer.” Why good: the line points directly to operational change (shorten pitch) and requires no interpretation; agent can act on it instantly.

arpu_90_days (positive): “ARPU ≈ 97.78 → recommend offers aligned to current spend and anchor price.” Why good: gives a concrete pricing cue tied to the customer’s spending, enabling consistent, data-aligned offers.

previous_classification = NEW_CLIENT (positive): “Onboarding message and starter incentives.” Why good: tells the agent the tone to use and a specific tactic (starter incentives), reducing decision time.

state_name (GUATEMALA, positive): “Mention local coverage/promotions when appropriate.” Why good: adds a short, localizable sentence to the script that can improve relevance with minimal cognitive load.